In [2]:
import os
import cv2
import insightface
from insightface.app import FaceAnalysis

# Initialize the model pack (e.g., buffalo_l)
app = FaceAnalysis(name='buffalo_l')
app.prepare(ctx_id=0, det_size=(640, 640))

# Resolve the image path relative to the project root (this notebook lives in FaceModel/)
PROJECT_ROOT = os.path.dirname(os.getcwd())
img_path = os.path.join(PROJECT_ROOT, 'pictures', 'DC1.jpg')

# Load an image
img = cv2.imread(img_path)
assert img is not None, f"cv2.imread failed for {img_path!r} — cwd is {os.getcwd()}"

# Detect and extract faces, landmarks, and 512-D embeddings
faces = app.get(img)
for face in faces:
  print(face.bbox, face.kps, face.embedding)

c:\Users\Bureau\OneDrive\Bureau\JamesBond AI\.venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


[165.35696 202.64957 678.8365  893.8178 ] [[294.74924 510.56354]
 [517.3016  465.12253]
 [423.64297 638.13513]
 [365.48828 750.30194]
 [541.59436 713.36536]] [ 6.10061809e-02 -1.68696892e+00  6.05740666e-01 -4.79236186e-01
  3.83379072e-01  2.58659899e-01 -7.41784200e-02 -1.63891196e+00
  1.13215792e+00  3.67279947e-01  1.13828909e+00 -4.29189235e-01
 -2.07253575e-01 -6.43712938e-01 -2.86049724e-01 -2.96088845e-01
 -9.46370959e-01  8.89666796e-01 -4.50872421e-01  1.88570634e-01
  5.20969868e-01  5.10025859e-01 -3.03047597e-01 -7.71238446e-01
 -2.17089489e-01 -1.00713146e+00 -5.36096931e-01 -1.88716555e+00
  3.97993505e-01  3.08244288e-01 -8.72218370e-01  5.93317449e-01
 -3.29107672e-01 -1.03789008e+00  1.13132215e+00 -4.35936391e-01
  1.89517200e-01 -6.57020450e-01  1.40717769e+00 -6.68211341e-01
 -5.07879496e-01 -6.37315154e-01  2.04615116e-01  1.05905557e+00
  5.15189707e-01 -2.99765021e-02 -1.08176601e+00 -7.61056840e-01
  4.44648743e-01 -1.40934527e+00  9.22652185e-01 -1.14874512e-

In [3]:
import os, cv2
import numpy as np
from insightface.app import FaceAnalysis

app = FaceAnalysis(name='buffalo_l')
app.prepare(ctx_id=0, det_size=(640, 640))

root = os.path.dirname(os.getcwd())
img_path = os.path.join(root, 'pictures', 'DC1.jpg')
img = cv2.imread(img_path)
assert img is not None, f"imread failed for {img_path!r} (cwd={os.getcwd()})"

faces = app.get(img)
print("image:", img.shape, "| faces detected:", len(faces))
for i, f in enumerate(faces):
    print(f"\n[{i}] bbox      : {f.bbox.astype(int).tolist()}")
    print(f"[{i}] score     : {f.det_score:.3f}")
    print(f"[{i}] kps       : {f.kps.astype(int).tolist()}")
    print(f"[{i}] embedding : dtype={f.embedding.dtype} shape={f.embedding.shape} norm={np.linalg.norm(f.embedding):.3f}")
    print(f"[{i}] emb[:6]   : {np.round(f.embedding[:6], 4).tolist()}")
    # bbox as x1,y1,x2,y2 -> width/height, useful for the "is it just the background" check
    x1, y1, x2, y2 = f.bbox
    print(f"[{i}] box size  : {int(x2-x1)}x{int(y2-y1)} px")


image: (1200, 941, 3) | faces detected: 1

[0] bbox      : [165, 202, 678, 893]
[0] score     : 0.871
[0] kps       : [[294, 510], [517, 465], [423, 638], [365, 750], [541, 713]]
[0] embedding : dtype=float32 shape=(512,) norm=20.571
[0] emb[:6]   : [0.061000000685453415, -1.687000036239624, 0.6057000160217285, -0.47920000553131104, 0.38339999318122864, 0.25870001316070557]
[0] box size  : 513x691 px


In [4]:
f = faces[0]
print("has normed_embedding:", hasattr(f, 'normed_embedding'))
if hasattr(f, 'normed_embedding'):
    n = f.normed_embedding
    print("normed: dtype=%s shape=%s norm=%.4f" % (n.dtype, n.shape, np.linalg.norm(n)))
print("\nraw  embedding norm :", np.linalg.norm(f.embedding))
print("normalized copy norm:", np.linalg.norm(f.embedding / np.linalg.norm(f.embedding)))

# cosine similarity of raw vs normalized against itself
a = f.embedding
print("\ncos(raw, raw)  =", float(a @ a / (np.linalg.norm(a)**2)))
print("dot(raw, raw)  =", float(a @ a), "  <- NOT a similarity, magnitude grows")


has normed_embedding: True
normed: dtype=float32 shape=(512,) norm=1.0000

raw  embedding norm : 20.570568
normalized copy norm: 0.99999994

cos(raw, raw)  = 0.9999999403953552
dot(raw, raw)  = 423.14825439453125   <- NOT a similarity, magnitude grows
